<a href="https://colab.research.google.com/github/ramanathanlab/genslm/blob/main/examples/generate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [11]:
!ls gdrive/MyDrive/patric_25m_epoch01-val_loss_0.57_bias_removed.pt
# This currently requires you to download the 25M model weights from Globus

gdrive/MyDrive/patric_25m_epoch01-val_loss_0.57_bias_removed.pt


In [1]:
import torch
from genslm import GenSLM
from Bio.Seq import Seq

In [2]:
# Load model
#model = GenSLM("genslm_25M_patric", model_cache_dir="/content/gdrive/MyDrive")
model = GenSLM("genslm_2.5B_patric", model_cache_dir="/lus/eagle/projects/FoundEpidem/xlian/genslm_models/2.5B")
model.eval()

# Select GPU device if it is available, else use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

2025-03-31 21:22:08.604577: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-31 21:22:44.725856: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


GenSLM(
  (model): GPTNeoXForCausalLM(
    (gpt_neox): GPTNeoXModel(
      (embed_in): Embedding(69, 3840)
      (emb_dropout): Dropout(p=0.0, inplace=False)
      (layers): ModuleList(
        (0-27): 28 x GPTNeoXLayer(
          (input_layernorm): LayerNorm((3840,), eps=1e-05, elementwise_affine=True)
          (post_attention_layernorm): LayerNorm((3840,), eps=1e-05, elementwise_affine=True)
          (post_attention_dropout): Dropout(p=0.0, inplace=False)
          (post_mlp_dropout): Dropout(p=0.0, inplace=False)
          (attention): GPTNeoXAttention(
            (query_key_value): Linear(in_features=3840, out_features=11520, bias=True)
            (dense): Linear(in_features=3840, out_features=3840, bias=True)
          )
          (mlp): GPTNeoXMLP(
            (dense_h_to_4h): Linear(in_features=3840, out_features=4096, bias=True)
            (dense_4h_to_h): Linear(in_features=4096, out_features=3840, bias=True)
            (act): FastGELUActivation()
          )
        )
 

In [41]:
vanA = 'atgattgaggtaatcatttcggcgatgcgcttggttgctcaggacatcattagccttgagtttgtccgggctgacggtggcttgcttccgcctgtcgaggccggcgcccacgtcgatgtgcatcttcctggcggcctgattcggcagtactcgctctggaatcaaccaggggcgcagagccattactgcatcggtgttctgaaggacccggcgtctcgtggtggttcgaaggcggtgcacgagaatcttcgcgtcgggatgcgcgtgcaaattagcgagccgaggaacctattcccattggaagagggggtggagcggagtctgctgttcgcgggcgggattggcattacgccgattctgtgtatggctcaagaattagcagcacgcgagcaagatttcgagttgcattattgcgcgcgttcgaccgaccgagcggcgttcgttgaatggcttaaggtttgcgactttgctgatcacgtacgtttccactttgacaatggcccggatcagcaaaaactgaatgccgcagcgctgctagcggccgaggccgaaggtacccacctttatgtctgtgggcccggcgggttcatggggcatgtgcttgataccgcgaaggagcagggctgggctgacaatcgactgcatcgagagtatttcgccgcggcgccgaatgtgagtgctgacgatggcagtttcgaggtgcggattcacagcaccggacaagtgcttcaggtccccgcggatcaaacggtctcccaggtgctcgatgcggccggaattatcgttcccgtttcttgtgagcagggcatctgcggtacttgcatcactcgggtggtagacggagagcctgatcatcgtgacttcttcctcacggatgcggagaaggcaaagaacgaccagttcaccccctgttgctcgcgagccaagagcgcctgtttggtcttggatctctaa'
vanA = vanA.upper()

if len(vanA) % 3 != 0:
    raise ValueError(f"DNA sequence length ({len(vanA)}) is not divisible by 3")
    
codons = [vanA[i:i+3] for i in range(0, len(vanA), 3)]

In [60]:
# Prompt the language model with a start codon
prom_len = 100
prom = codons[:prom_len]
prompt = model.tokenizer.encode(' '.join(prom), return_tensors="pt").to(device)

tokens = model.model.generate(
    prompt,
    max_length= 400 - prom_len,  # Increase this to generate longer sequences
    min_length= 330 - prom_len,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    num_return_sequences=4,  # Change the number of sequences to generate
    remove_invalid_values=True,
    use_cache=True,
    pad_token_id=model.tokenizer.encode("[PAD]")[0],
    temperature=1.0,
)

sequences = model.tokenizer.batch_decode(tokens, skip_special_tokens=True)
print('generated')

generated


In [61]:
def translate_codon_sequences(sequences):
    proteins = []
    for seq_str in sequences:
        # Remove spaces and create a Biopython Seq object
        dna_seq = Seq(seq_str.replace(" ", "").upper())
        protein = dna_seq.translate(to_stop=True)
        proteins.append(str(protein))
    return proteins

proteins = translate_codon_sequences(sequences)
print('Generated by genslm 2.5B model (not fine-tuned)')
for i in proteins:
    print(i)

Generated by genslm 2.5B model (not fine-tuned)
MIEVIISAMRLVAQDIISLEFVRADGGLLPPVEAGAHVDVHLPGGLIRQYSLWNQPGAQSHYCIGVLKDPASRGGSKAVHENLRVGMRVQISEPRNLFPLVPATEYVLLAGGIGITPLLSMAYRLQSLGDVFDLHFYAREAAQAGFVDFVREGAFAARLQFHTDDGDALLTLEAPDAHSALNTEVIWSSLPREVILARMSKQQKLWAQGDRFFSDQFQVDADRAHRHHRVFHIQAKNQGRSLVISASGQQLPIKAGESVLLADGTNHAIPCGDCGYLWQRRQQAAQLRFVLESHLGQWGV
MIEVIISAMRLVAQDIISLEFVRADGGLLPPVEAGAHVDVHLPGGLIRQYSLWNQPGAQSHYCIGVLKDPASRGGSKAVHENLRVGMRVQISEPRNLFPLSQAAAHTVLVAGGIGITPLVAMAYWQADTRVDFSAQLYSGVAPGAMLYVNAEALRASYGPRDFFHCTDVRPEAGGAFALVQGLAATLQPGEVYLCGPAGLVQAVRAGYAQRDLPKLAVHFEYFAAAPVRPAQPFDIRFEVHSSELPVVTELISTGIAERFDCEIALACESGVCGTCITKIIDGTPEHRDMFLCDDEKACG
MIEVIISAMRLVAQDIISLEFVRADGGLLPPVEAGAHVDVHLPGGLIRQYSLWNQPGAQSHYCIGVLKDPASRGGSKAVHENLRVGMRVQISEPRNLFPLAKEPFQTLLYAGGIGITPILAMAAKLGRTQHAFHLHYYSARHARAAFYSELNQKFGNAVTFYDQERMMEMDLRGVLPLGTARDGAHVLVCGPTGFMAAIESAAQLLGRSPASLHYEQFAPAAMPGQGDGTFYLQVGSTGRILEVMADCDAKPVQAHTEIDHALGALELTRMSASLPATEAHCTIELPDARKFLDTMEAHY
MIEVIISAMRLVAQDIISLEFVRADGGLLPPVEAGAHVDVHLPGGLIRQ

In [58]:
print('vanA segment prompt (translated)', Seq(''.join(codons[:prom_len])).translate(), '', sep = '\n')
print('full vanA', Seq(''.join(codons)).translate(), sep = '\n')

vanA segment prompt (translated)
MIEVIISAMRLVAQDIISLEFVRADGGLLPPVEAGAHVDVHLPGGLIRQYSLWNQPGAQSHYCIGVLKDPASRGGSKAVHENLRVGMRVQISEPRNLFPL

full vanA
MIEVIISAMRLVAQDIISLEFVRADGGLLPPVEAGAHVDVHLPGGLIRQYSLWNQPGAQSHYCIGVLKDPASRGGSKAVHENLRVGMRVQISEPRNLFPLEEGVERSLLFAGGIGITPILCMAQELAAREQDFELHYCARSTDRAAFVEWLKVCDFADHVRFHFDNGPDQQKLNAAALLAAEAEGTHLYVCGPGGFMGHVLDTAKEQGWADNRLHREYFAAAPNVSADDGSFEVRIHSTGQVLQVPADQTVSQVLDAAGIIVPVSCEQGICGTCITRVVDGEPDHRDFFLTDAEKAKNDQFTPCCSRAKSACLVLDL*


In [25]:
prompt = model.tokenizer.encode('AAA AAA', return_tensors="pt").to(device)
prompt

tensor([[16, 16]], device='cuda:0')